# Diabetes Prediction using Machine Learning
**Presented By**
- Devansh Rathore (23FE10CSE00204)
- Arnav Verma (23FE10CSE00191)

---

## 1. Introduction

With the rapid rise of lifestyle diseases and urbanization, **diabetes** has become one of the most critical public health challenges of the 21st century. According to the International Diabetes Federation, over **500 million people** globally live with diabetes — and nearly **half remain undiagnosed**.

**Diabetes** is a chronic metabolic disorder characterized by elevated blood sugar (glucose) levels, resulting from the body's inability to produce or effectively use insulin. If left undetected, it can lead to severe complications including:
- Cardiovascular disease
- Kidney failure
- Blindness
- Nerve damage

This project focuses on building a **Diabetes Early Detection System** using machine learning, which predicts whether a patient is likely to have diabetes based on clinical diagnostic measurements.

### Objectives:
- Collect and preprocess the Pima Indians Diabetes dataset from Kaggle
- Analyze feature distributions and inter-feature correlations
- Train and compare multiple ML models (Logistic Regression, Decision Tree, KNN, Random Forest)
- Evaluate model performance using standard classification metrics
- Identify the most impactful clinical features for diabetes prediction

### Challenges:
- Zero-value entries in clinical features (physiologically impossible)
- Class imbalance between diabetic and non-diabetic patients
- Non-linear relationships between clinical features and outcome
- Need for high recall to minimize false negatives in medical diagnosis

## 2. Contributions

### 2.1 Multi-Model Diabetes Prediction
Four machine learning models — Logistic Regression, Decision Tree, K-Nearest Neighbors, and Random Forest — were implemented and compared to identify the best-performing approach for diabetes classification.

### 2.2 Data Preprocessing and Cleaning
The dataset was preprocessed by replacing physiologically impossible zero values with median imputation, applying feature scaling for distance-based models, and performing an 80/20 stratified train-test split.

### 2.3 Visualization and Analysis
Comprehensive EDA using pair plots, correlation heatmaps, histograms, and feature importance charts to understand the data and model behavior.

## 3. Problem Statement

**To develop a machine learning system that predicts the onset of diabetes in patients based on clinical diagnostic measurements, enabling early intervention and improved healthcare outcomes.**

### Scope
- Applicable in hospital screening systems and smart health platforms
- Useful for primary care physicians as a clinical decision support tool
- Supports public health agencies in identifying high-risk populations

### Relevance
- Enables early detection before clinical symptoms appear
- Reduces diagnostic cost compared to laboratory testing
- Saves lives through timely medical intervention

### AQI-Style Classification of Diabetes Risk

| Class | Outcome | Meaning |
|-------|---------|----------|
| 0 | Non-Diabetic | Glucose and other markers within normal range |
| 1 | Diabetic | Clinical indicators suggest diabetes onset |

## 4. Dataset Description

The **Pima Indians Diabetes Database** is sourced from the **National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK)** and is available on Kaggle.

**Dataset URL:** https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

| Feature | Description |
|---------|-------------|
| Pregnancies | Number of times pregnant |
| Glucose | Plasma glucose concentration (2-hr oral glucose tolerance test) |
| BloodPressure | Diastolic blood pressure (mm Hg) |
| SkinThickness | Triceps skin fold thickness (mm) |
| Insulin | 2-Hour serum insulin (mu U/ml) |
| BMI | Body Mass Index (weight in kg / height in m²) |
| DiabetesPedigreeFunction | Genetic risk function based on family history |
| Age | Age in years |
| **Outcome** | **Target: 1 = Diabetic, 0 = Non-Diabetic** |

- **Total Records:** 768
- **Features:** 8 independent clinical variables
- **Target:** Binary classification (Outcome)
- **All patients:** Female, at least 21 years old, of Pima Indian heritage

## 5. Algorithm Used — Random Forest Classifier

**Random Forest Classifier** is an ensemble learning algorithm that combines multiple decision trees to improve prediction accuracy and reduce overfitting. It is the primary model used in this project.

### Working Principle

1. **Bootstrap Sampling** — Multiple subsets of training data are created by random sampling with replacement.
2. **Training Multiple Decision Trees** — Each subset trains an independent decision tree.
3. **Random Feature Selection** — At each split, a random subset of features is considered.
4. **Majority Voting** — The final prediction is determined by majority vote across all trees:

$$\hat{y} = \text{mode}\left(\{y_1, y_2, \ldots, y_N\}\right)$$

### Advantages
- Reduces overfitting compared to a single decision tree
- Handles non-linear relationships effectively
- Provides feature importance scores
- Robust to noise and outliers

### Limitations
- Computationally more expensive than simpler models
- Less interpretable (black-box nature)
- Requires hyperparameter tuning for optimal results

## 6. Step-by-Step Process to Predict Diabetes

### 6.1 Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay)

print('All libraries imported successfully!')

### 6.2 Loading the Dataset

In [ ]:
# Load the dataset from Kaggle
# To use in Colab: upload the file or use kaggle API
# data = pd.read_csv('/content/drive/MyDrive/diabetes.csv')

# For demonstration, loading from local path:
data = pd.read_csv('diabetes.csv')

print('Dataset Shape:', data.shape)
print('\nFirst 5 rows:')
print(data.head())
print('\nData Types:')
print(data.dtypes)
print('\nMissing Values:', data.isnull().sum().sum())

### 6.3 Data Preprocessing

In [ ]:
# Replace biologically impossible zero values with median
# Glucose, BloodPressure, SkinThickness, Insulin, BMI cannot be 0

cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols_with_zeros:
    median_val = data[col].median()
    data[col] = data[col].replace(0, median_val)
    print(f'{col}: replaced zeros with median = {median_val:.2f}')

print('\nPreprocessing complete. Dataset shape:', data.shape)

### 6.4 Exploratory Data Analysis (EDA)

In [ ]:
# 1. Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = data['Outcome'].value_counts()
axes[0].pie(counts.values, labels=['No Diabetes', 'Diabetes'],
            autopct='%1.1f%%', colors=['#065A82', '#E05C5C'],
            startangle=140, textprops={'fontsize': 13})
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')

# 2. Glucose Distribution by Class
for outcome, label, color in [(0, 'No Diabetes', '#065A82'), (1, 'Diabetes', '#E05C5C')]:
    subset = data[data['Outcome'] == outcome]['Glucose']
    axes[1].hist(subset, bins=25, alpha=0.6, label=label, color=color, edgecolor='white')
axes[1].set_xlabel('Glucose Level'); axes[1].set_ylabel('Count')
axes[1].set_title('Glucose Distribution by Outcome', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
fig, ax = plt.subplots(figsize=(9, 7))
corr = data.corr().round(2)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop correlations with Outcome:')
print(corr['Outcome'].sort_values(ascending=False))

In [ ]:
# Pairplot for key features
key_features = ['Glucose', 'BMI', 'Age', 'DiabetesPedigreeFunction', 'Outcome']
sns.pairplot(data[key_features], hue='Outcome',
             palette={0: '#065A82', 1: '#E05C5C'}, plot_kws={'alpha': 0.5})
plt.suptitle('Pair Plot of Key Features', y=1.01, fontsize=14, fontweight='bold')
plt.show()

### 6.5 Feature Engineering

In [ ]:
# Select features and target
X = data.drop('Outcome', axis=1)
y = data['Outcome']

print('Feature matrix shape:', X.shape)
print('Target vector shape:', y.shape)
print('\nFeature statistics:')
print(X.describe().round(2))

### 6.6 Train-Test Split

In [ ]:
# 80/20 stratified split to maintain class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling for distance-based models
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print('Training set size:', X_train.shape)
print('Test set size:', X_test.shape)
print('\nClass distribution in training set:')
print(y_train.value_counts())

### 6.7 Model Training

In [ ]:
# --- Logistic Regression ---
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_sc, y_train)
print('Logistic Regression trained.')

# --- Decision Tree ---
dt = DecisionTreeClassifier(random_state=42, max_depth=5)
dt.fit(X_train, y_train)
print('Decision Tree trained.')

# --- K-Nearest Neighbors ---
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
print('K-Nearest Neighbors trained.')

# --- Random Forest Classifier (Primary Model) ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print('Random Forest trained.')

### 6.8 Model Evaluation

In [ ]:
models = {
    'Logistic Regression': (lr, X_test_sc),
    'Decision Tree': (dt, X_test),
    'K-Nearest Neighbors': (knn, X_test_sc),
    'Random Forest': (rf, X_test)
}

results = {}
print(f"{'Model':<25} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1 Score':>10} {'ROC-AUC':>9}")
print('-' * 75)

for name, (model, X_eval) in models.items():
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'y_pred': y_pred
    }
    r = results[name]
    print(f"{name:<25} {r['Accuracy']:>9.4f} {r['Precision']:>10.4f} {r['Recall']:>8.4f} {r['F1 Score']:>10.4f} {r['ROC-AUC']:>9.4f}")

### 6.9 Plotting Results

In [ ]:
# Accuracy Comparison
fig, ax = plt.subplots(figsize=(9, 5))
names = list(results.keys())
accs = [results[n]['Accuracy'] * 100 for n in names]
colors = ['#1C7293', '#065A82', '#028090', '#02C39A']
bars = ax.bar(names, accs, color=colors, edgecolor='white', linewidth=1.5)
ax.bar_label(bars, fmt='%.1f%%', padding=5, fontsize=11, fontweight='bold')
ax.set_ylim(60, 105)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted — Random Forest
rf_pred = results['Random Forest']['y_pred']
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(y_test.values[:80], label='Actual', color='#065A82',
        linewidth=1.5, marker='o', markersize=4)
ax.plot(rf_pred[:80], label='Predicted', color='#E05C5C',
        alpha=0.8, linewidth=1.5, linestyle='--', marker='s', markersize=4)
ax.set_title('Actual vs Predicted — Random Forest (First 80 test samples)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Outcome')
ax.set_yticks([0, 1])
ax.set_yticklabels(['No Diabetes', 'Diabetes'])
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(fi.index, fi.values, color='#065A82')
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Feature Importances — Random Forest', fontsize=14, fontweight='bold')
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_xlim(0, fi.values.max() * 1.18)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, rf_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Diabetes', 'Diabetes'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Observations and Conclusion

### Evaluation Metrics — Random Forest Classifier

| Metric | Value |
|--------|-------|
| Accuracy | 0.7792 |
| Precision | 0.8346 |
| Recall | 0.8908 |
| F1 Score | 0.8618 |
| ROC-AUC | 0.7595 |

### Observations
- **Glucose** is the most important feature for diabetes prediction, with a correlation of ~0.47 with the outcome.
- **BMI** and **DiabetesPedigreeFunction** are also strong predictors, highlighting the roles of obesity and genetic predisposition.
- **Logistic Regression** achieved the highest accuracy (81.17%) among all models, demonstrating that the dataset has a reasonably linear decision boundary.
- **Random Forest** and **KNN** tied on accuracy (77.92%) but Random Forest had higher AUC (0.7595 vs 0.7324), indicating better probabilistic discrimination.
- The **high recall (0.89)** of Random Forest means it correctly identifies most diabetic patients — critical for minimizing dangerous false negatives in medical screening.

### Conclusion
This project demonstrates that machine learning models can effectively predict diabetes from clinical measurements. With strong recall and AUC scores, **Random Forest Classifier** proves well-suited for medical screening tasks. **Logistic Regression** emerged as the overall best performer in terms of accuracy, confirming that interpretable models can compete with ensemble methods on structured clinical data.

### Future Improvements
- Integration with real-time patient health monitoring systems
- Inclusion of additional features: HbA1c, diet patterns, physical activity
- Hyperparameter tuning using GridSearchCV / RandomizedSearchCV
- Exploration of advanced models: XGBoost, LightGBM, and Deep Neural Networks
- SMOTE oversampling to address class imbalance
- Deployment as a REST API or mobile health application